# 第三讲：matplotlib 金融可视化 —— K线图、收益率分布、多子图与中文

**学习目标**
- 解决 matplotlib 中文乱码，掌握 macOS 字体配置的完整链路
- 用纯 matplotlib 手绘 K 线图，理解 OHLC 数据结构和绘制原理
- 从收益率 histogram、密度曲线到 Q-Q 图，建立「分布诊断」的完整视角
- 掌握 `subplots` / `GridSpec` 布局体系，能搭建专业的多面板金融仪表盘

**环境依赖**

本讲需要以下库（已在当前环境就绪）：
```bash
pip install numpy pandas matplotlib yfinance scipy seaborn
```

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import matplotlib.font_manager as fm
from matplotlib.gridspec import GridSpec

import yfinance as yf
from scipy import stats
import seaborn as sns

# 固定随机种子，保证结果可复现
np.random.seed(42)

print("✓ 所有库导入成功")

---

## 3.1 中文显示配置 —— 先解决「口口口」

这是每个中文用户都会踩的坑。matplotlib 默认用 DejaVu Sans 渲染文字，它没有 CJK 字形，所以中文全部变成方块。解决路径有四步：

### 3.1.1 诊断：当前环境有哪些字体可用？

In [ ]:
# 查找系统中所有可用的中文字体
chinese_kw = ['PingFang', 'Heiti', 'Songti', 'Kaiti', 'Libian', 'Baoli', 
              'Noto.*CJK', 'WenQuanYi', 'SimHei', 'SimSun', 'Microsoft YaHei']

import re
all_fonts = fm.fontManager.ttflist
print(f"matplotlib 已索引 {len(all_fonts)} 个字体\n")

found = []
for f in all_fonts:
    for kw in chinese_kw:
        if re.search(kw, f.name, re.IGNORECASE):
            found.append(f)
            break

if found:
    print("可用的中文字体：")
    for f in found:
        print(f"  {f.name:30s} → {f.fname}")
else:
    print("⚠️ 未在 matplotlib 字体缓存中找到中文字体——需要手动注册")
    # macOS 系统字体通常在这些位置
    import glob
    system_fonts = glob.glob('/System/Library/Fonts/**/*.tt[cf]', recursive=True)
    system_fonts += glob.glob('/System/Library/AssetsV2/**/PingFang.ttc', recursive=True)
    for path in system_fonts:
        print(f"  系统路径: {path}")

### 3.1.2 方案 A：全局配置（推荐给个人项目）

设置一次，整个 notebook 所有图表自动使用中文。

核心是这三行：
```python
plt.rcParams['font.sans-serif'] = ['PingFang SC', 'Heiti SC', ...]  # 字体回退列表
plt.rcParams['axes.unicode_minus'] = False                           # 解决负号显示为方块
```

> **原理**：`axes.unicode_minus = False` 告诉 matplotlib 用 ASCII 连字符 `-` 渲染负号，而不是 Unicode 减号 `−`（U+2212），因为很多中文字体没有这个字形。

In [ ]:
# === 全局中文字体配置（只执行一次） ===
plt.rcParams['font.sans-serif'] = [
    'PingFang SC',    # 苹方（macOS 默认中文字体）
    'Heiti SC',       # 黑体-简
    'STHeiti',        # 华文黑体
    'sans-serif'      # 最后的 fallback
]
plt.rcParams['axes.unicode_minus'] = False  # 负号正常显示

# 验证
fig, ax = plt.subplots(figsize=(6, 1.5))
ax.text(0.5, 0.5, '中文测试 · 收益率% · −0.05 · 沪深300', 
        transform=ax.transAxes, ha='center', va='center', fontsize=14)
ax.set_title('matplotlib 中文显示验证')
ax.axis('off')
plt.show()

### 3.1.3 方案 B：局部字体设置（不改全局 rcParams）

当你不想「污染」全局设置时，每个图表单独指定字体。`font_manager.FontProperties` 是最灵活的方式。

In [ ]:
# 局部字体方案：不影响全局 rcParams
cn_font = fm.FontProperties(fname='/System/Library/Fonts/STHeiti Medium.ttc', size=14)
cn_font_small = fm.FontProperties(fname='/System/Library/Fonts/STHeiti Medium.ttc', size=10)

fig, ax = plt.subplots()
ax.set_title('局部字体设置演示', fontproperties=cn_font)
ax.set_xlabel('时间（交易日）', fontproperties=cn_font_small)
ax.set_ylabel('价格（元）', fontproperties=cn_font_small)
ax.plot([1, 2, 3], [10, 12, 9])
plt.show()

print("注意：全局 rcParams 未被修改，其他图表仍用默认字体")

### 3.1.4 字体回退链（Fallback chain）

`font.sans-serif` 是一个**列表**，matplotlib 按顺序尝试：找到第一个能渲染目标字符的字体就用它。

```python
plt.rcParams['font.sans-serif'] = [
    'PingFang SC',   # ① 首选：macOS 原生的高品质中文字体
    'Heiti SC',      # ② 备选：黑体
    'Arial'          # ③ 终局 fallback：英文/数字
]
```

这样「苹果股价 180.50 元」中的中文走 PingFang，数字和英文走 Arial——各取所长。

---

## 3.2 K 线图 —— 手绘每个蜡烛

很多教程直接调 `mplfinance`，你只知道「画出来了」但不知道为什么。这一节我们**用最原始的 matplotlib Rectangle + Line2D 手绘 K 线**，彻底理解 OHLC → 蜡烛的映射关系。

> 完成后再看一眼 `mplfinance` 的做法，你会觉得它不过是封装了一层循环。

### 3.2.1 数据准备：从 yfinance 拉取真实行情

In [ ]:
# 下载贵州茅台（600519.SS）最近 3 个月的日线数据
# yfinance 中上海交易所加 .SS，深圳加 .SZ
ticker = '600519.SS'
df = yf.download(ticker, period='3mo', interval='1d', progress=False)

# yfinance 2.x 返回 MultiIndex 列，flatten 一下
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

print(f"数据维度: {df.shape}")
print(f"日期范围: {df.index[0].strftime('%Y-%m-%d')} → {df.index[-1].strftime('%Y-%m-%d')}")
df.head(8)

### 3.2.2 核心映射：OHLC → 蜡烛

一根蜡烛由两部分构成：

```
         ┬  High（上影线顶端）
         │
    ┌────┴────┐
    │  实体    │  ← Close > Open 则为红色（阳线），反之为绿色（阴线）
    └────┬────┘
         │
         ┴  Low（下影线底端）
```

- **实体矩形**：底边 = min(Open, Close)，顶边 = max(Open, Close)，宽度 = 1 天的 60%~80%
- **影线**：从实体中心向上到 High，向下到 Low 的竖线
- **颜色**：中国习惯 红涨绿跌；国际惯例 绿涨红跌。本节用中国习惯。

In [ ]:
def draw_candlestick_manual(ax, df, width=0.6, colorup='red', colordown='green'):
    """
    用 matplotlib Rectangle + Line2D 逐根绘制 K 线。
    
    参数
    ----
    ax : matplotlib.axes.Axes
        目标坐标轴
    df : pandas.DataFrame
        必须包含 'Open', 'High', 'Low', 'Close' 列，index 为日期
    width : float
        实体宽度（以交易日为单位，0.6 表示实体占 60% 的日区间）
    colorup : str
        阳线（Close > Open）颜色
    colordown : str
        阴线（Close < Open）颜色
    """
    # 将日期索引转为数值坐标
    dates = mdates.date2num(df.index.to_pydatetime())
    
    for i, (date, row) in enumerate(zip(dates, df.itertuples())):
        open_p, high_p, low_p, close_p = row.Open, row.High, row.Low, row.Close
        
        # 判断涨跌
        if close_p >= open_p:
            color = colorup
            body_bottom = open_p     # 实体底部 = 开盘价
            body_height = close_p - open_p  # 实体高度 = 收盘-开盘
        else:
            color = colordown
            body_bottom = close_p
            body_height = open_p - close_p
        
        # 1. 画影线（从 Low 到 High 的竖线）
        ax.plot([date, date], [low_p, high_p], color=color, linewidth=0.8)
        
        # 2. 画实体（矩形）
        rect = mpatches.Rectangle(
            (date - width / 2, body_bottom),  # 左下角坐标
            width, body_height,               # 宽度、高度
            facecolor=color if body_height > 0 else 'none',  # 阳线实心，平盘空心
            edgecolor=color,
            linewidth=0.8
        )
        ax.add_patch(rect)
    
    # 设置日期格式
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')


# 取最近 30 个交易日来画，太密看不清
df_subset = df.iloc[-30:].copy()

fig, ax = plt.subplots(figsize=(14, 6))
draw_candlestick_manual(ax, df_subset, width=0.6, colorup='#DC143C', colordown='#228B22')
ax.set_title(f'{ticker} 贵州茅台 —— 最近 {len(df_subset)} 个交易日 K 线（手绘）', fontsize=14)
ax.set_ylabel('价格（元）', fontsize=12)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

### 3.2.3 叠加成交量 —— 做一幅「专业级」K 线图

真正的行情软件都是 K 线在上、成交量在下。我们用 `GridSpec` 做出不等高的双面板布局。

In [ ]:
def draw_candlestick_with_volume(df, title='K线 + 成交量', days=60):
    """
    画出 K 线（上） + 成交量柱（下）的专业双面板图。
    """
    df_plot = df.iloc[-days:].copy()
    dates = mdates.date2num(df_plot.index.to_pydatetime())
    
    # 构建 GridSpec：上 3 份（K线），下 1 份（成交量），共享 X 轴
    fig = plt.figure(figsize=(16, 8))
    gs = GridSpec(4, 1, figure=fig, hspace=0.05, height_ratios=[3, 3, 3, 1])
    ax_k = fig.add_subplot(gs[:-1, 0])   # K 线占上面 3 行
    ax_v = fig.add_subplot(gs[-1, 0])     # 成交量占最后 1 行
    
    # --- 上：K 线 ---
    for i, (date, row) in enumerate(zip(dates, df_plot.itertuples())):
        open_p, high_p, low_p, close_p = row.Open, row.High, row.Low, row.Close
        if close_p >= open_p:
            color, bottom, height = '#DC143C', open_p, close_p - open_p
        else:
            color, bottom, height = '#228B22', close_p, open_p - close_p
        
        ax_k.plot([date, date], [low_p, high_p], color=color, lw=0.8)
        rect = mpatches.Rectangle((date - 0.3, bottom), 0.6, max(height, 0.01),
                                  facecolor=color if height > 0 else 'none',
                                  edgecolor=color, lw=0.8)
        ax_k.add_patch(rect)
    
    # 叠加 5 日和 20 日均线
    if len(df_plot) >= 20:
        ax_k.plot(dates, df_plot['Close'].rolling(5).mean(), 'orange', lw=1, label='MA5')
        ax_k.plot(dates, df_plot['Close'].rolling(20).mean(), 'purple', lw=1, label='MA20')
        ax_k.legend(loc='upper left')
    
    ax_k.set_title(title, fontsize=14, fontweight='bold')
    ax_k.set_ylabel('价格', fontsize=11)
    ax_k.grid(True, alpha=0.25)
    ax_k.tick_params(labelbottom=False)  # 隐藏 K 线图的 X 轴标签
    
    # --- 下：成交量 ---
    vol_colors = ['#DC143C' if row.Close >= row.Open else '#228B22' 
                  for row in df_plot.itertuples()]
    ax_v.bar(dates, df_plot['Volume'].values, width=0.6, color=vol_colors, alpha=0.7)
    ax_v.set_ylabel('成交量', fontsize=11)
    ax_v.grid(True, alpha=0.25)
    
    # 格式化成交量（亿股 / 万手）
    ax_v.yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f'{x/1e6:.0f}M' if x >= 1e6 else f'{x/1e3:.0f}K'))
    
    # 共享 X 轴日期格式
    for ax in [ax_k, ax_v]:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
        ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
    plt.setp(ax_v.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    fig.tight_layout()
    return fig


fig = draw_candlestick_with_volume(df, title=f'{ticker} 贵州茅台 —— K 线 + 成交量 + 均线', days=60)
plt.show()

### 3.2.4 理解：为什么手绘比 `mplfinance` 更有学习价值？

| 维度 | 手绘 | `mplfinance` |
|------|------|-------------|
| 绘制机制 | 你精确控制每根蜡烛的 Rectangle + Line2D | 封装好的 `make_addplot` |
| 自定义能力 | 无限制——你可以在实体上画箭头、加标注、改透明度 | 受 API 限制，复杂定制很痛苦 |
| 学习成本 | 一次写完，终身理解 | 每次都要查文档 |
| 适用场景 | 量化研究、策略可视化、论文图表 | 快速浏览行情 |

当你需要画出「在突破位置标注箭头 + 基本面事件文字」这样的定制图时，手绘方案就是唯一的武器了。

---

## 3.3 收益率分布 —— 数据长什么样？

金融中的第一个问题是：「这个资产的收益率服从正态分布吗？」

我们用三种递增的诊断工具来回答：
1. **直方图 + 核密度估计**：看大致形状
2. **叠加正态分布**：看偏离程度
3. **Q-Q 图**：精确诊断尾部行为

### 3.3.1 计算收益率

> **为什么用对数收益率？**
> 
> 简单收益率 $R_t = \frac{P_t - P_{t-1}}{P_{t-1}}$ 不可加——多期累积是 $(1+R_1)(1+R_2)...-1$，而取对数后 $r_t = \ln(P_t/P_{t-1})$ 变为加法：$\sum r_t = \ln(P_T/P_0)$。加法比乘法好处理得多，且对数收益率更接近正态分布。

In [ ]:
# 对数收益率
returns = np.log(df['Close'] / df['Close'].shift(1)).dropna()

print(f"交易日数: {len(returns)}")
print(f"日收益率均值: {returns.mean():.6f}  ({returns.mean()*100:.4f}%)")
print(f"日收益率标准差: {returns.std():.6f}  ({returns.std()*100:.4f}%)")
print(f"偏度 (skewness): {returns.skew():.4f}")
print(f"峰度 (kurtosis): {returns.kurtosis():.4f}  (正态分布 = 0)")
print(f"Jarque-Bera 统计量: {stats.jarque_bera(returns)[0]:.2f}")
print(f"Jarque-Bera p 值:   {stats.jarque_bera(returns)[1]:.6f}")
print()

if stats.jarque_bera(returns)[1] < 0.05:
    print("结论：p < 0.05，拒绝正态分布假设——收益率不是正态的 ✓")
else:
    print("结论：无法拒绝正态分布假设")

### 3.3.2 直方图 + 核密度估计（KDE）+ 正态分布叠加

这是金融论文中最常见的「收益分布三件套」。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- 左图：直方图 + KDE + 正态分布叠加 ---
ax = axes[0]

# 1) 直方图（密度归一化，使得总面积为 1）
counts, bins, patches = ax.hist(returns, bins=40, density=True, 
                                 alpha=0.5, color='steelblue', edgecolor='white',
                                 label='收益率直方图')

# 2) 核密度估计（seaborn 的 KDE 比 scipy 的 gaussian_kde 更好看）
sns.kdeplot(returns, ax=ax, color='darkblue', lw=2.5, label='核密度估计 (KDE)')

# 3) 叠加正态分布曲线（用样本均值和标准差）
x = np.linspace(returns.min(), returns.max(), 500)
mu, sigma = returns.mean(), returns.std()
norm_pdf = stats.norm.pdf(x, mu, sigma)
ax.plot(x, norm_pdf, 'r--', lw=2, label=f'正态分布 N({mu*100:.2f}%, {sigma*100:.2f}%)')

ax.set_title('收益率分布：直方图 + KDE + 正态对比', fontsize=13)
ax.set_xlabel('对数日收益率', fontsize=11)
ax.set_ylabel('密度', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

# --- 右图：Q-Q 图（Quantile-Quantile Plot）---
ax = axes[1]
stats.probplot(returns, dist='norm', plot=ax)
ax.set_title('Q-Q 图：收益率 vs 正态分布', fontsize=13)
ax.set_xlabel('理论分位数（正态）', fontsize=11)
ax.set_ylabel('样本分位数', fontsize=11)
ax.grid(True, alpha=0.2)

# 解读标注
ax.annotate('尾部偏离 →', xy=(2.5, 0.03), xytext=(2.0, 0.025),
            fontsize=10, color='darkred',
            arrowprops=dict(arrowstyle='->', color='darkred'))

fig.suptitle(f'{ticker} 贵州茅台 —— 收益率分布诊断', fontsize=15, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

### 3.3.3 如何解读 Q-Q 图？

Q-Q 图是诊断尾部行为最锋利的工具：

| 观察 | 含义 |
|------|------|
| 点紧贴 45° 对角线 | 数据服从正态分布 |
| **两端向上翘**（偏离对角线） | **右偏 + 厚尾**（正收益的极端值比正态预测的更多），常见于成长股、科技股 |
| **两端向下弯** | **左偏**（负收益的极端值更多），常见于危机期 |
| **S 形** | 数据比正态更「集中」，尾部比正态更薄 |

绝大部分股票收益率都是「**两端上翘**」——正收益和负收益的极端值都比正态分布预测的更频繁出现。这也是为什么用正态分布做 VaR 会严重低估风险。

### 3.3.4 对比多只股票的收益率分布

In [ ]:
# 同时拉取茅台、招行、宁德时代
tickers = {
    '600519.SS': '贵州茅台',
    '600036.SS': '招商银行', 
    '300750.SZ': '宁德时代'
}

returns_dict = {}
for code, name in tickers.items():
    data = yf.download(code, period='6mo', interval='1d', progress=False)
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)
    r = np.log(data['Close'] / data['Close'].shift(1)).dropna()
    returns_dict[name] = r
    print(f"{name:8s}  均值={r.mean()*100:.3f}%  std={r.std()*100:.3f}%  "
          f"偏度={r.skew():+.3f}  峰度={r.kurtosis():+.3f}")

# 三只股票 KDE 叠加对比
fig, ax = plt.subplots(figsize=(14, 6))
colors = ['#DC143C', '#228B22', '#FF8C00']
for (name, r), c in zip(returns_dict.items(), colors):
    sns.kdeplot(r, ax=ax, color=c, lw=2.5, label=f'{name} (σ={r.std()*100:.2f}%)')

ax.set_title('三只 A 股收益率 KDE 对比', fontsize=14, fontweight='bold')
ax.set_xlabel('对数日收益率', fontsize=12)
ax.set_ylabel('密度', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.2)
fig.tight_layout()
plt.show()

---

## 3.4 多子图布局 —— 从 `subplots` 到 `GridSpec`

当一张图放不下所有信息时，你需要多子图。matplotlib 提供了两个层级的布局工具：

| 层级 | API | 适用场景 |
|------|-----|---------|
| **简单等分** | `plt.subplots(nrows, ncols)` | 2×2 分析面板、m×n 网格 |
| **不等分/异形** | `GridSpec` | K线+成交量、宽窄不一的混合布局 |
| **子图内嵌** | `add_axes` / `inset_axes` | 局部放大图 |

### 3.4.1 基础等分布局：`plt.subplots()`

最常用的 2×2 分析面板——用四个子图同时展示价格、收益率、波动率和成交量。

In [ ]:
df_6m = df.copy()
df_6m['Return'] = np.log(df_6m['Close'] / df_6m['Close'].shift(1))
df_6m['Volatility'] = df_6m['Return'].rolling(20).std() * np.sqrt(252)  # 年化波动率
df_6m = df_6m.dropna()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
# axes 是 2×2 的 numpy 数组，axes[0,0] 是左上角

# ① 收盘价走势
ax = axes[0, 0]
ax.plot(df_6m.index, df_6m['Close'], color='#DC143C', lw=1.2)
ax.fill_between(df_6m.index, df_6m['Close'].min(), df_6m['Close'], 
                alpha=0.1, color='#DC143C')
ax.set_title('① 收盘价走势', fontsize=13, fontweight='bold')
ax.set_ylabel('价格（元）')
ax.grid(True, alpha=0.2)

# ② 日收益率
ax = axes[0, 1]
colors = ['#DC143C' if r >= 0 else '#228B22' for r in df_6m['Return']]
ax.bar(range(len(df_6m)), df_6m['Return'], color=colors, width=0.8, alpha=0.7)
ax.axhline(y=0, color='black', lw=0.5)
ax.set_title('② 日收益率（对数）', fontsize=13, fontweight='bold')
ax.set_ylabel('收益率')
ax.grid(True, alpha=0.2)
ax.set_xticks([])

# ③ 20 日滚动年化波动率
ax = axes[1, 0]
ax.plot(df_6m.index, df_6m['Volatility'], color='steelblue', lw=1.5)
ax.fill_between(df_6m.index, 0, df_6m['Volatility'], alpha=0.15, color='steelblue')
ax.set_title('③ 20 日滚动年化波动率', fontsize=13, fontweight='bold')
ax.set_ylabel('年化波动率')
ax.grid(True, alpha=0.2)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# ④ 成交量
ax = axes[1, 1]
vol_colors = ['#DC143C' if row.Close >= row.Open else '#228B22' 
              for row in df_6m.itertuples()]
ax.bar(df_6m.index, df_6m['Volume'], color=vol_colors, width=0.8, alpha=0.6)
ax.set_title('④ 成交量', fontsize=13, fontweight='bold')
ax.set_ylabel('成交量')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
ax.grid(True, alpha=0.2)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

fig.suptitle(f'{ticker} 贵州茅台 —— 四维分析面板', fontsize=16, fontweight='bold', y=1.01)
fig.tight_layout()
plt.show()

### 3.4.2 `GridSpec` 的高级不等分布局

`subplots` 只能等分。当你需要「左边一个宽图 + 右边两个窄图」或者「上面大图 + 下面三个小图」的时候，`GridSpec` 是正解。

下面是一个**左侧宽面板（走势+收益）+ 右侧两窄面板（分布+统计）**的布局：

In [ ]:
fig = plt.figure(figsize=(18, 9))

# 定义网格：4 行 × 6 列
#   col 0-3 (宽) → 左侧大区域 (0:3, 0:4)
#   col 4-5 (窄) → 右侧区域
gs = GridSpec(4, 6, figure=fig, hspace=0.35, wspace=0.4)

# --- 左侧：价格走势（占 3 行 × 4 列） ---
ax1 = fig.add_subplot(gs[0:3, 0:4])
ax1.plot(df_6m.index, df_6m['Close'], color='#DC143C', lw=1.2)
ax1.set_title('收盘价走势（宽面板）', fontsize=13, fontweight='bold')
ax1.set_ylabel('价格（元）')
ax1.grid(True, alpha=0.2)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

# --- 左侧底部：日收益率（占 1 行 × 4 列） ---
ax2 = fig.add_subplot(gs[3, 0:4], sharex=ax1)
colors_bar = ['#DC143C' if r >= 0 else '#228B22' for r in df_6m['Return']]
ax2.bar(df_6m.index, df_6m['Return'], color=colors_bar, width=0.8, alpha=0.7)
ax2.axhline(y=0, color='black', lw=0.5)
ax2.set_title('日收益率', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.2)

# --- 右上：收益分布直方图 + KDE ---
ax3 = fig.add_subplot(gs[0:2, 4:6])
ax3.hist(df_6m['Return'], bins=35, density=True, alpha=0.5, 
         color='steelblue', edgecolor='white')
sns.kdeplot(df_6m['Return'], ax=ax3, color='darkblue', lw=2)
x_range = np.linspace(df_6m['Return'].min(), df_6m['Return'].max(), 200)
ax3.plot(x_range, stats.norm.pdf(x_range, df_6m['Return'].mean(), df_6m['Return'].std()),
         'r--', lw=1.5, alpha=0.7)
ax3.set_title('收益分布', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.2)

# --- 右下：统计摘要表格 ---
ax4 = fig.add_subplot(gs[2:4, 4:6])
ax4.axis('off')

# 构建统计表
stats_data = [
    ['均值', f"{df_6m['Return'].mean()*100:.3f}%"],
    ['标准差', f"{df_6m['Return'].std()*100:.3f}%"],
    ['偏度', f"{df_6m['Return'].skew():.4f}"],
    ['峰度', f"{df_6m['Return'].kurtosis():.4f}"],
    ['夏普比率', f"{df_6m['Return'].mean()/df_6m['Return'].std()*np.sqrt(252):.3f}"],
    ['最大回撤', f"{(df_6m['Close'] / df_6m['Close'].cummax() - 1).min()*100:.2f}%"],
    ['胜率', f"{(df_6m['Return'] > 0).mean()*100:.1f}%"],
    ['JB p-value', f"{stats.jarque_bera(df_6m['Return'])[1]:.4f}"],
]

table = ax4.table(cellText=stats_data, colLabels=['指标', '数值'],
                  loc='center', cellLoc='center',
                  colWidths=[0.35, 0.35])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.5)
ax4.set_title('统计摘要', fontsize=13, fontweight='bold', y=1.05)

fig.suptitle('GridSpec 不等分布局实战：综合仪表盘', fontsize=16, fontweight='bold', y=1.01)
plt.show()

### 3.4.3 子图内嵌：局部放大（Inset Axes）

有时候你想在一个图里嵌套另一个小图作为局部放大——比如在主走势图里放一个小窗展示最近一周的详情。

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# 主图：全时段收盘价
ax.plot(df_6m.index, df_6m['Close'], color='#DC143C', lw=1.2)
ax.set_title('收盘价走势（全时段）+ 最近 15 日局部放大', fontsize=13, fontweight='bold')
ax.set_ylabel('价格（元）')
ax.grid(True, alpha=0.2)

# 高亮最近 15 天的区域
recent_days = 15
ax.axvspan(df_6m.index[-recent_days], df_6m.index[-1], 
           alpha=0.12, color='orange', label=f'最近 {recent_days} 日')
ax.legend()

# 内嵌小图：最近 15 天的放大
inset_ax = ax.inset_axes([0.55, 0.15, 0.4, 0.4])  # [left, bottom, width, height] 相对于主图的分数
inset_ax.plot(df_6m.index[-recent_days:], df_6m['Close'].iloc[-recent_days:], 
              color='#DC143C', lw=2, marker='o', markersize=4)
inset_ax.set_title(f'最近 {recent_days} 日放大', fontsize=10)
inset_ax.grid(True, alpha=0.3)
inset_ax.tick_params(labelsize=8)
inset_ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.setp(inset_ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 画连接线（从主图区域到小图）
ax.indicate_inset_zoom(inset_ax, edgecolor='gray', alpha=0.6)

plt.show()

### 3.4.4 实践技巧：这些参数值得记住

```python
# 共享坐标轴（价格/时间对齐）
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)   # 共享 X 轴
fig, (ax1, ax2) = plt.subplots(1, 2, sharey=True)   # 共享 Y 轴

# 自定义子图间距
fig.subplots_adjust(hspace=0.3, wspace=0.3)          # 行间距、列间距
fig.tight_layout(pad=2.0)                             # 自动压缩，pad 控制边缘

# GridSpec 中的跨度
ax = fig.add_subplot(gs[0:2, 1:3])   # 占第 0-1 行，第 1-2 列
ax = fig.add_subplot(gs[2, :])        # 占第 2 行，所有列
ax = fig.add_subplot(gs[:, 0])        # 占所有行，第 0 列

# 内嵌子图的精确位置
inset = ax.inset_axes([x, y, w, h])   # 分数坐标 [0,1]
```

---

## 3.5 综合实战：完整的股票分析仪表盘

前面学到的所有技术集中到一个图中——六个面板，三种布局方式，展示一只股票的全面画像。

**仪表盘包含**：
1. K线图 + 均线（左上）
2. 成交量（左中）
3. 收益率分布（右上）
4. Q-Q 图（右中）
5. 滚动波动率（左下）
6. 累计收益曲线（右下）

In [ ]:
# 准备数据
df_dash = df.iloc[-90:].copy()
df_dash['Return'] = np.log(df_dash['Close'] / df_dash['Close'].shift(1))
df_dash['CumReturn'] = df_dash['Return'].cumsum()
df_dash['Vol20'] = df_dash['Return'].rolling(20).std() * np.sqrt(252)
df_dash = df_dash.dropna()

dates = mdates.date2num(df_dash.index.to_pydatetime())
returns_clean = df_dash['Return'].dropna()

# --- 构建 3×2 仪表盘 ---
fig = plt.figure(figsize=(20, 14))
gs = GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35,
              height_ratios=[2, 2, 1.5])

# ① K线图（左上：2行 × 2列）
ax1 = fig.add_subplot(gs[0:2, 0:2])
for i, (date, row) in enumerate(zip(dates, df_dash.itertuples())):
    open_p, high_p, low_p, close_p = row.Open, row.High, row.Low, row.Close
    if close_p >= open_p:
        color, bottom, height = '#DC143C', open_p, close_p - open_p
    else:
        color, bottom, height = '#228B22', close_p, open_p - close_p
    ax1.plot([date, date], [low_p, high_p], color=color, lw=0.7)
    rect = mpatches.Rectangle((date - 0.3, bottom), 0.6, max(height, 0.01),
                              facecolor=color if height > 0 else 'none',
                              edgecolor=color, lw=0.7)
    ax1.add_patch(rect)
ax1.plot(dates, df_dash['Close'].rolling(5).mean(), 'orange', lw=1, label='MA5')
ax1.plot(dates, df_dash['Close'].rolling(20).mean(), 'purple', lw=1, label='MA20')
ax1.set_title('① K线图 + 均线', fontsize=13, fontweight='bold')
ax1.set_ylabel('价格')
ax1.legend(loc='upper left', fontsize=8)
ax1.grid(True, alpha=0.2)
ax1.tick_params(labelbottom=False)

# ② 收益率分布 + KDE（右上：1行 × 1列）
ax2 = fig.add_subplot(gs[0, 2])
ax2.hist(returns_clean, bins=30, density=True, alpha=0.45, 
         color='steelblue', edgecolor='white')
sns.kdeplot(returns_clean, ax=ax2, color='darkblue', lw=2)
x_r = np.linspace(returns_clean.min(), returns_clean.max(), 200)
ax2.plot(x_r, stats.norm.pdf(x_r, returns_clean.mean(), returns_clean.std()),
         'r--', lw=1.5, alpha=0.7)
ax2.set_title('② 收益分布 + 正态对比', fontsize=13, fontweight='bold')
ax2.set_xlabel('对数收益率')
ax2.grid(True, alpha=0.2)

# ③ Q-Q 图（右中：1行 × 1列）
ax3 = fig.add_subplot(gs[1, 2])
stats.probplot(returns_clean, dist='norm', plot=ax3)
ax3.set_title('③ Q-Q 图', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.2)
ax3.get_lines()[0].set_markersize(4)
ax3.get_lines()[1].set_color('red')

# ④ 成交量（左下底：占 1行 × 2列）
ax4 = fig.add_subplot(gs[2, 0:2], sharex=ax1)
vol_c = ['#DC143C' if r >= 0 else '#228B22' for r in df_dash['Return']]
ax4.bar(dates, df_dash['Volume'].values, width=0.6, color=vol_c, alpha=0.6)
ax4.set_title('④ 成交量', fontsize=13, fontweight='bold')
ax4.set_ylabel('成交量')
ax4.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
ax4.grid(True, alpha=0.2)
ax4.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.setp(ax4.xaxis.get_majorticklabels(), rotation=45, ha='right')

# ⑤ 滚动波动率 + 累计收益（右下底：1行 × 1列，双 Y 轴）
ax5 = fig.add_subplot(gs[2, 2])
color_vol = 'steelblue'
color_cum = '#DC143C'
ax5.plot(df_dash.index, df_dash['Vol20'], color=color_vol, lw=1.5, label='年化波动率')
ax5.set_ylabel('年化波动率', color=color_vol, fontsize=10)
ax5.tick_params(axis='y', labelcolor=color_vol)

ax5b = ax5.twinx()  # 第二个 Y 轴
ax5b.plot(df_dash.index, df_dash['CumReturn'], color=color_cum, lw=1.5, 
          label='累计收益', alpha=0.8)
ax5b.set_ylabel('累计对数收益', color=color_cum, fontsize=10)
ax5b.tick_params(axis='y', labelcolor=color_cum)
ax5b.axhline(y=0, color='gray', lw=0.5, ls='--')

ax5.set_title('⑤ 波动率 + 累计收益', fontsize=13, fontweight='bold')
ax5.grid(True, alpha=0.2)
ax5.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.setp(ax5.xaxis.get_majorticklabels(), rotation=45, ha='right')

fig.suptitle(f'{ticker} 贵州茅台 —— 六维综合仪表盘', 
             fontsize=17, fontweight='bold', y=1.01)
plt.show()

print("仪表盘包含：K线+均线 | 成交量 | 收益分布 | Q-Q图 | 滚动波动率 | 累计收益")

---

## 本讲小结

| 章节 | 你学会了什么 | 核心 API |
|------|------------|---------|
| **3.1 中文配置** | 字体回退链、全局 vs 局部设置、负号修复 | `rcParams['font.sans-serif']`, `FontProperties` |
| **3.2 K线图** | OHLC → 蜡烛的逐根绘制，叠加成交量/均线 | `matplotlib.patches.Rectangle`, `matplotlib.dates` |
| **3.3 收益率分布** | 直方图/KDE/正态叠加/Q-Q 图的诊断体系 | `sns.kdeplot`, `stats.probplot`, `stats.jarque_bera` |
| **3.4 多子图** | `subplots` 等分、`GridSpec` 异形、`inset_axes` 内嵌 | `GridSpec`, `inset_axes`, `twinx`, `sharex` |
| **3.5 实战** | 六个面板的综合仪表盘（K线+成交量+分布+Q-Q+波动率+收益） | 以上所有技术的组合 |

### 延伸方向

- **交互式图表**：用 `plotly` 或 `bokeh` 替代 matplotlib，实现悬停查看、缩放、时间滑块
- **mplfinance**：如果你经常需要标准化的 K 线图，`pip install mplfinance` 是最快方案
- **matplotlib 动画**：用 `FuncAnimation` 做出随时间推进的 K 线动画
- **风格化**：`plt.style.use('seaborn-v0_8-darkgrid')` 可以一键切换图表风格

---

*第三讲完 · 2026-06-02*